In [23]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

# Dark theme template for all Plotly figures
DARK_TEMPLATE = 'plotly_dark'
DARK_BG = '#111111'
LIGHT_TEXT = '#FFFFFF'

# Exploratory Data Analysis (EDA)
## Mental Health & Lifestyle Factors Study
*Comprehensive analysis of depression severity and health-related variables*

In [4]:
df = pd.read_csv('data.csv')

print('\nDataset Loaded Successfully')
print('=' * 80)
print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('=' * 80)
df.head()


Dataset Loaded Successfully
Dataset Shape: 5,000 rows x 26 columns


,id,country,average_sunny_days,age,gender,employment_status,marital_status,education_level,monthly_income,loneliness,...,high_coffee_consumption,ultra_processed_food,low_dietary_diversity,alcohol_consumption,smoking,poor_sleep_quality,anxiety_level,social_media_use_hours,news_hours_per_day,depression_severity
0,user_075722,Italy,230,26,Other,Student,Married,Bachelor,1000–2000€,Occasional,...,False,False,False,False,False,False,NaN,10.0,2.6,Mild
1,user_080185,Italy,230,64,Other,Student,Widowed,High School,2000–3000€,NaN,...,False,False,False,False,True,True,NaN,7.7,2.2,Moderate
2,user_019865,Spain,250,54,Male,Unemployed,Single,Bachelor,>5000€,NaN,...,False,True,True,True,False,False,NaN,8.4,3.8,Mild
3,user_076700,Sweden,170,51,Male,Retired,Divorced,Master,1000–2000€,NaN,...,True,False,False,True,True,True,Moderate,4.2,1.3,Moderate
4,user_092992,France,200,71,Other,Student,Married,Master,2000–3000€,Frequent,...,False,False,True,True,False,True,NaN,2.6,1.8,Severe


In [5]:
rows, cols = df.shape
print('\nDATASET OVERVIEW')
print('=' * 80)
print(f'Total Records  : {rows:,}')
print(f'Total Features : {cols}')
print('\nColumn Names and Types:')
print('-' * 80)

for i, col in enumerate(df.columns, 1):
    dtype = str(df[col].dtype)
    non_null = df[col].notna().sum()
    missing = df[col].isna().sum()
    print(f'{i:2}. {col:<30} | Type: {dtype:<12} | Non-Null: {non_null:>6} | Missing: {missing:>4}')


DATASET OVERVIEW
Total Records  : 5,000
Total Features : 26

Column Names and Types:
--------------------------------------------------------------------------------
 1. id                             | Type: object       | Non-Null:   5000 | Missing:    0
 2. country                        | Type: object       | Non-Null:   5000 | Missing:    0
 3. average_sunny_days             | Type: int64        | Non-Null:   5000 | Missing:    0
 4. age                            | Type: int64        | Non-Null:   5000 | Missing:    0
 5. gender                         | Type: object       | Non-Null:   5000 | Missing:    0
 6. employment_status              | Type: object       | Non-Null:   5000 | Missing:    0
 7. marital_status                 | Type: object       | Non-Null:   5000 | Missing:    0
 8. education_level                | Type: object       | Non-Null:   5000 | Missing:    0
 9. monthly_income                 | Type: object       | Non-Null:   5000 | Missing:    0
10. loneliness

In [6]:
print('\nDATA TYPE CLASSIFICATION')
print('=' * 80)

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f'\nNumerical Features ({len(numeric_cols)}):' )
for col in numeric_cols:
    print(f'  {col}')

print(f'\nCategorical Features ({len(categorical_cols)}):' )
for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f'  {col:<32} - {unique_count} unique values')


DATA TYPE CLASSIFICATION

Numerical Features (4):
  average_sunny_days
  age
  social_media_use_hours
  news_hours_per_day

Categorical Features (10):
  id                               - 5000 unique values
  country                          - 6 unique values
  gender                           - 3 unique values
  employment_status                - 4 unique values
  marital_status                   - 4 unique values
  education_level                  - 4 unique values
  monthly_income                   - 5 unique values
  loneliness                       - 2 unique values
  anxiety_level                    - 3 unique values
  depression_severity              - 3 unique values


In [7]:
# Feature type distribution visualization
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Feature Type Distribution', 'All Features by Type'),
    specs=[[{'type':'pie'}, {'type':'bar'}]])

type_counts = {'Numerical': len(numeric_cols), 'Categorical': len(categorical_cols)}
fig.add_trace(
    go.Pie(labels=list(type_counts.keys()), 
            values=list(type_counts.values()),
            marker=dict(colors=['#1f77b4', '#ff7f0e']),
            hole=0.3),
    row=1, col=1
)

all_cols_with_types = [(col, 'Numerical') for col in numeric_cols] + \
                      [(col, 'Categorical') for col in categorical_cols]
col_names = [c[0] for c in all_cols_with_types]
col_colors = ['#1f77b4' if c[1] == 'Numerical' else '#ff7f0e' for c in all_cols_with_types]

fig.add_trace(
    go.Bar(y=col_names, x=[1]*len(col_names),
           marker=dict(color=col_colors),
           orientation='h',
           showlegend=False,
           hovertemplate='%{y}<extra></extra>'),
    row=1, col=2
)

fig.update_layout(height=500, title_text='Feature Type Analysis', 
                  title_font_size=14, showlegend=True, template=DARK_TEMPLATE)
fig.update_xaxes(showticklabels=False, row=1, col=2)
fig.show()

In [8]:
print('\nSTATISTICAL SUMMARY - Numerical Features')
print('=' * 80)

stats_summary = df[numeric_cols].describe().round(3)
print(stats_summary.to_string())

print('\nAdditional Statistics:')
print('-' * 80)
for col in numeric_cols:
    skewness = df[col].skew()
    kurtosis_val = df[col].kurtosis()
    cv = (df[col].std() / df[col].mean() * 100) if df[col].mean() != 0 else 0
    print(f'{col:<25} | Skewness: {skewness:>7.3f} | Kurtosis: {kurtosis_val:>7.3f} | CV: {cv:>6.2f}%')


STATISTICAL SUMMARY - Numerical Features
       average_sunny_days       age  social_media_use_hours  news_hours_per_day
count            5000.000  5000.000                5000.000            5000.000
mean              193.648    49.027                   4.954               1.994
std                37.097    18.143                   2.890               1.159
min               150.000    18.000                   0.000               0.000
25%               160.000    33.000                   2.400               1.000
50%               200.000    49.000                   5.000               2.000
75%               230.000    64.000                   7.400               3.000
max               250.000    80.000                  10.000               4.000

Additional Statistics:
--------------------------------------------------------------------------------
average_sunny_days        | Skewness:   0.325 | Kurtosis:  -1.458 | CV:  19.16%
age                       | Skewness:  -0.002 | Kurto

In [9]:
print('\nCATEGORICAL FEATURES - Value Distribution')
print('=' * 80)

for col in categorical_cols:
    unique_vals = df[col].dropna().unique()
    value_counts = df[col].value_counts()
    print(f'\n{col.upper()} ({len(unique_vals)} unique values)')
    print('-' * 80)
    for val, count in value_counts.items():
        pct = (count / len(df) * 100)
        bar_length = int(pct / 2)
        print(f'  {val:<25} : {count:>6} ({pct:>5.1f}%) {"=" * bar_length}')


CATEGORICAL FEATURES - Value Distribution

ID (5000 unique values)
--------------------------------------------------------------------------------
  user_041657               :      1 (  0.0%) 
  user_075722               :      1 (  0.0%) 
  user_080185               :      1 (  0.0%) 
  user_019865               :      1 (  0.0%) 
  user_076700               :      1 (  0.0%) 
  user_092992               :      1 (  0.0%) 
  user_076435               :      1 (  0.0%) 
  user_084005               :      1 (  0.0%) 
  user_034918               :      1 (  0.0%) 
  user_065567               :      1 (  0.0%) 
  user_044260               :      1 (  0.0%) 
  user_019932               :      1 (  0.0%) 
  user_068095               :      1 (  0.0%) 
  user_006154               :      1 (  0.0%) 
  user_092608               :      1 (  0.0%) 
  user_085010               :      1 (  0.0%) 
  user_031998               :      1 (  0.0%) 
  user_077231               :      1 (  0.0%) 
  use

In [10]:
print('\nMISSING VALUES ANALYSIS')
print('=' * 80)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percent': missing_pct
}).sort_values('Missing_Count', ascending=False)

missing_df_display = missing_df[missing_df['Missing_Count'] > 0]

if len(missing_df_display) > 0:
    print('\nColumns with Missing Values:')
    print('-' * 80)
    for col, row in missing_df_display.iterrows():
        print(f'{col:<30} : {int(row["Missing_Count"]):>6} missing ({row["Missing_Percent"]:>5.1f}%)')
    print(f'\nTotal Missing Cells: {missing.sum():,}')
    print(f'Columns Affected: {(missing > 0).sum()} out of {len(df.columns)}')
    print(f'Overall Sparsity: {(missing.sum() / (len(df) * len(df.columns)) * 100):.2f}%')
else:
    print('No missing values detected')


MISSING VALUES ANALYSIS

Columns with Missing Values:
--------------------------------------------------------------------------------
loneliness                     :   1646 missing ( 32.9%)
anxiety_level                  :   1304 missing ( 26.1%)

Total Missing Cells: 2,950
Columns Affected: 2 out of 26
Overall Sparsity: 2.27%


In [11]:
# Missing values visualization
cols_with_missing = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=True)

if len(cols_with_missing) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Missing Values Count', 'Missing Values Percentage'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}]]
    )

    fig.add_trace(
        go.Bar(x=cols_with_missing['Missing_Count'], 
               y=cols_with_missing.index,
               orientation='h',
               marker=dict(color='#d62728'),
               name='Count',
               hovertemplate='%{y}: %{x}<extra></extra>'),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(x=cols_with_missing['Missing_Percent'], 
               y=cols_with_missing.index,
               orientation='h',
               marker=dict(color='#ff7f0e'),
               name='Percentage',
               hovertemplate='%{y}: %{x}%<extra></extra>'),
        row=1, col=2
    )

    fig.update_xaxes(title_text='Count', row=1, col=1)
    fig.update_xaxes(title_text='Percentage (%)', row=1, col=2)
    fig.update_layout(height=400, title_text='Missing Values Distribution',
                      title_font_size=14, showlegend=False, template=DARK_TEMPLATE)
    fig.show()
else:
    print('No visualization needed')

In [12]:
print('\nTARGET VARIABLE ANALYSIS - Depression Severity')
print('=' * 80)

depression = df['depression_severity'].dropna()
depression_counts = df['depression_severity'].value_counts().sort_index()

print(f'\nDepression Severity Distribution:')
print('-' * 80)
for severity, count in depression_counts.items():
    pct = (count / len(depression) * 100)
    bar_length = int(pct / 2)
    print(f'  {severity:<12} : {count:>6} ({pct:>5.1f}%) {"=" * bar_length}')

print(f'\nSummary Statistics:')
print(f'  Total Responses      : {len(depression):,}')
print(f'  Missing Values       : {df["depression_severity"].isna().sum()}')
print(f'  Most Common Severity : {depression_counts.idxmax()}')
print(f'  Distribution         : {dict(depression_counts)}')


TARGET VARIABLE ANALYSIS - Depression Severity

Depression Severity Distribution:
--------------------------------------------------------------------------------
  Mild         :   1630 ( 32.6%) ================
  Moderate     :   1657 ( 33.1%) ================
  Severe       :   1713 ( 34.3%) =================

Summary Statistics:
  Total Responses      : 5,000
  Missing Values       : 0
  Most Common Severity : Severe
  Distribution         : {'Mild': np.int64(1630), 'Moderate': np.int64(1657), 'Severe': np.int64(1713)}


In [13]:
print('\nCORRELATION ANALYSIS')
print('=' * 80)

numeric_df = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
corr_matrix = numeric_df.corr()

print('\nPearson Correlation Matrix:')
print(corr_matrix.round(3).to_string())

print('\nNote: Depression severity is categorical. For analysis, use encoding.')


CORRELATION ANALYSIS

Pearson Correlation Matrix:
                        average_sunny_days    age  social_media_use_hours  news_hours_per_day
average_sunny_days                   1.000 -0.007                  -0.010              -0.003
age                                 -0.007  1.000                  -0.030               0.039
social_media_use_hours              -0.010 -0.030                   1.000              -0.015
news_hours_per_day                  -0.003  0.039                  -0.015               1.000

Note: Depression severity is categorical. For analysis, use encoding.


In [14]:
# Depression severity distribution visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Severity Distribution', 'Pie Chart', 'Cumulative Distribution', 'Severity Counts'),
    specs=[[{'type': 'histogram'}, {'type': 'pie'}],
           [{'type': 'scatter'}, {'type': 'bar'}]]
)

severity_colors = {'Mild': '#90EE90', 'Moderate': '#FFD700', 'Severe': '#FF6B6B'}

depression_counts_sorted = df['depression_severity'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=depression_counts_sorted.index, y=depression_counts_sorted.values,
           marker=dict(color=[severity_colors.get(x, '#808080') for x in depression_counts_sorted.index]),
           name='Count',
           hovertemplate='%{x}: %{y}<extra></extra>'),
    row=2, col=2
)

fig.add_trace(
    go.Pie(labels=depression_counts_sorted.index, values=depression_counts_sorted.values,
           marker=dict(colors=[severity_colors.get(x, '#808080') for x in depression_counts_sorted.index]),
           hovertemplate='%{label}: %{value} (%{percent})<extra></extra>'),
    row=1, col=2
)

sorted_vals = df['depression_severity'].value_counts().sort_index()
cumsum = sorted_vals.cumsum() / sorted_vals.sum()
fig.add_trace(
    go.Scatter(x=sorted_vals.index, y=cumsum.values,
               mode='lines+markers',
               name='Cumulative',
               line=dict(color='#1f77b4', width=2),
               marker=dict(size=8),
               hovertemplate='%{x}: %{y:.1%}<extra></extra>'),
    row=2, col=1
)

numerical_depression = df[numeric_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)
fig.add_trace(
    go.Histogram(x=numerical_depression, nbinsx=20,
                 marker=dict(color='#1f77b4'),
                 name='Distribution',
                 hovertemplate='Range: %{x}<extra></extra>'),
    row=1, col=1
)

fig.update_layout(height=700, title_text='Depression Severity Analysis',
                  title_font_size=14, showlegend=False, template=DARK_TEMPLATE)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Severity Count', row=2, col=2)
fig.update_yaxes(title_text='Cumulative Proportion', row=2, col=1)
fig.show()

In [15]:
import plotly.express as px

# Visualize top categorical features vs depression severity
cat_cols_to_plot = [col for col in categorical_cols if col not in ['id', 'depression_severity']][:6]

fig = make_subplots(rows=2, cols=3,
    subplot_titles=cat_cols_to_plot)

severity_colors = {'Mild': '#90EE90', 'Moderate': '#FFD700', 'Severe': '#FF6B6B'}
severity_order = ['Mild', 'Moderate', 'Severe']

for idx, col in enumerate(cat_cols_to_plot):
    row, col_pos = divmod(idx, 3)
    cross = pd.crosstab(df[col], df['depression_severity'], normalize='index') * 100
    for sev in severity_order:
        if sev in cross.columns:
            fig.add_trace(go.Bar(
                name=sev,
                x=cross.index.astype(str),
                y=cross[sev],
                marker_color=severity_colors[sev],
                showlegend=(idx == 0),
                hovertemplate=f'%{{x}}<br>{sev}: %{{y:.1f}}%<extra></extra>'
            ), row=row+1, col=col_pos+1)

fig.update_layout(
    barmode='stack',
    title='Categorical Features vs Depression Severity (%)',
    height=600,
    template=DARK_TEMPLATE
)
fig.show()


In [16]:
# Correlation heatmap
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text:.2f}',
    textfont={"size": 9},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='Correlation Heatmap - Numerical Features',
    height=600,
    width=700,
    template=DARK_TEMPLATE
)
fig.show()

In [17]:
print('\nCATEGORICAL FEATURES ANALYSIS')
print('=' * 80)

for col in categorical_cols:
    if col != 'depression_severity' and col != 'id':
        depression_by_cat = pd.crosstab(df[col], df['depression_severity'], normalize='index') * 100
        print(f'\n{col.upper()} vs Depression Severity:')
        print('-' * 80)
        print(depression_by_cat.round(1).to_string())


CATEGORICAL FEATURES ANALYSIS

COUNTRY vs Depression Severity:
--------------------------------------------------------------------------------
depression_severity  Mild  Moderate  Severe
country                                    
France               32.0      31.8    36.2
Germany              29.4      32.4    38.2
Italy                38.5      30.8    30.7
Netherlands          24.8      35.4    39.8
Spain                40.9      34.6    24.4
Sweden               29.7      33.6    36.7

GENDER vs Depression Severity:
--------------------------------------------------------------------------------
depression_severity  Mild  Moderate  Severe
gender                                     
Female               32.5      33.5    34.0
Male                 32.7      32.4    34.8
Other                32.5      33.5    34.0

EMPLOYMENT_STATUS vs Depression Severity:
--------------------------------------------------------------------------------
depression_severity  Mild  Moderate  Severe
em

In [18]:
print('\nRISK FACTORS ANALYSIS')
print('=' * 80)

lifestyle_cols = [col for col in categorical_cols if col not in ['id', 'country', 'gender', 'employment_status', 
                                                                   'marital_status', 'education_level', 'depression_severity']]

print('\nLifestyle Risk Factors Distribution by Depression Severity:')
print('-' * 80)

for col in lifestyle_cols[:10]:
    cross_tab = pd.crosstab(df[col], df['depression_severity'], margins=True)
    print(f'\n{col}:')
    print(cross_tab.to_string())


RISK FACTORS ANALYSIS

Lifestyle Risk Factors Distribution by Depression Severity:
--------------------------------------------------------------------------------

monthly_income:
depression_severity  Mild  Moderate  Severe   All
monthly_income                                   
1000–2000€            323       321     349   993
2000–3000€            311       338     339   988
3000–5000€            329       321     324   974
<1000€                335       364     370  1069
>5000€                332       313     331   976
All                  1630      1657    1713  5000

loneliness:
depression_severity  Mild  Moderate  Severe   All
loneliness                                       
Frequent              347       520     811  1678
Occasional            626       538     512  1676
All                   973      1058    1323  3354

anxiety_level:
depression_severity  Mild  Moderate  Severe   All
anxiety_level                                    
Mild                  576       399    

In [21]:
print('\n' + '='*80)
print('EXPLORATORY DATA ANALYSIS - COMPREHENSIVE SUMMARY')
print('='*80)

print(f"""
DATASET OVERVIEW
{'='*80}
Total Records             : {df.shape[0]:,}
Total Features            : {df.shape[1]}
Numerical Features        : {len(numeric_cols)}
Categorical Features      : {len(categorical_cols)}
Missing Values            : {df.isnull().sum().sum():,} ({(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100):.2f}% sparsity)

TARGET VARIABLE: DEPRESSION SEVERITY
{'='*80}
Distribution:
""")

depression_dist = df['depression_severity'].value_counts()
for severity in ['Mild', 'Moderate', 'Severe']:
    if severity in depression_dist.index:
        count = depression_dist[severity]
        pct = (count / len(df['depression_severity'].dropna()) * 100)
        bar = '=' * int(pct / 5)
        print(f'  {severity:<12}: {count:>6} ({pct:>5.1f}%) {bar:<20}')

print(f"""
Most Common       : {df['depression_severity'].value_counts().idxmax()}
Missing Values    : {df['depression_severity'].isna().sum()}

KEY INSIGHTS & PATTERNS
{'='*80}
Numerical Features Statistics:
""")

for col in numeric_cols:
    print(f'  {col:<25}: mean={df[col].mean():>8.2f}, std={df[col].std():>8.2f}, range=[{df[col].min():>8.2f}, {df[col].max():>8.2f}]')

print(f"""
Top Categorical Features:
""")

for i, col in enumerate(categorical_cols[:5], 1):
    unique = df[col].nunique()
    print(f'  {i}. {col:<30}: {unique} unique values')

print(f"""
DATA QUALITY ASSESSMENT
{'='*80}
Completeness         : {((1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100):.1f}%
Columns with Missing : {(df.isnull().sum() > 0).sum()}
Duplicate Rows       : {df.duplicated().sum()}
Data Types Detected  : {df.dtypes.nunique()} different types

""")
print('='*80)


EXPLORATORY DATA ANALYSIS - COMPREHENSIVE SUMMARY

DATASET OVERVIEW
Total Records             : 5,000
Total Features            : 26
Numerical Features        : 4
Categorical Features      : 10
Missing Values            : 2,950 (2.27% sparsity)

TARGET VARIABLE: DEPRESSION SEVERITY
Distribution:

  Mild        :   1630 ( 32.6%) ======              
  Moderate    :   1657 ( 33.1%) ======              
  Severe      :   1713 ( 34.3%) ======              

Most Common       : Severe
Missing Values    : 0

KEY INSIGHTS & PATTERNS
Numerical Features Statistics:

  average_sunny_days       : mean=  193.65, std=   37.10, range=[  150.00,   250.00]
  age                      : mean=   49.03, std=   18.14, range=[   18.00,    80.00]
  social_media_use_hours   : mean=    4.95, std=    2.89, range=[    0.00,    10.00]
  news_hours_per_day       : mean=    1.99, std=    1.16, range=[    0.00,     4.00]

Top Categorical Features:

  1. id                            : 5000 unique values
  2. countr